# Assignment 4 — One- and Two-Sample Tests

## Group members

- Bibek Chaudhary (344326)
- Samo Susa (343876)
- Sneha Koirala (345788)
- Tomas Psotka (343207)

## Table of contents

1. [Imports and display settings](#imports)
2. [Part 1 — One-sample decisions](#part-1)
3. [Part 2 — Independent coating processes](#part-2)
4. [Part 3 — Paired building-energy measurements](#part-3)
5. [Part 4 — Comparing two independent proportions](#part-4)
6. [Part 5 — AI-use and conclusion](#part-5)

<a id="imports"></a>

## Imports and display settings

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

In [16]:
data_dir = Path("data") if Path("data").exists() else Path("../data")
sensor = pd.read_csv(data_dir / "assignment03_sensor_calibration.csv")
access = pd.read_csv(data_dir / "assignment03_access_control.csv")
sensor.head() 


errors = sensor["calibration_error_c"]
n = errors.count()
sample_mean = errors.mean()
sample_std = errors.std(ddof=1)

print("Sample size:", n)
print("Sample mean:", round(sample_mean, 4), "°C")
print("Sample standard deviation:", round(sample_std, 4), "°C")

print("Shape:", access.shape)
print("Columns:", access.columns.tolist())

Sample size: 64
Sample mean: 0.1883 °C
Sample standard deviation: 0.6257 °C
Shape: (240, 2)
Columns: ['attempt_id', 'false_reject']


<a id="part-1"></a>

## Part 1 — One-sample decisions

### Mean Sensor Calibration Error

Use assignment03_sensor_calibration.csv. The engineering team asks whether population mean calibration error differs from zero; errors with absolute mean below  0.50 degree celsius are regarded as small in practice.

### a.

State the null hypothesis (H₀), the two-sided (H₁), and (α = 0.05). Use a one-sample t-procedure and report the sample mean, test statistic, degrees of freedom, and two-sided  p-value.


**Answer (a)**

We want to investigate whether the population mean sensor calibration error differs from zero.

**Null hypothesis:**

$$H_0: \mu = 0$$

**Alternative hypothesis:**

$$H_1: \mu \neq 0$$

**Significance level:**

$$\alpha = 0.05$$



In [14]:

t_stat, p_value = stats.ttest_1samp(
    errors,
    popmean=0,
    alternative="two-sided"
)

df = n - 1

print(f"Sample mean: {sample_mean:.4f} °C")
print(f"Test statistic: {t_stat:.4f}")
print(f"Degrees of freedom: {df}")
print(f"Two-sided p-value: {p_value:.6f}")


Sample mean: 0.1883 °C
Test statistic: 2.4069
Degrees of freedom: 63
Two-sided p-value: 0.019031


**Interpretation:**

We use a **two-sided one-sample t-test** because we are testing one population mean against zero, the population standard deviation is unknown, and we are interested in deviations in either direction.

The test statistic was calculated using:

$$
t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}
$$

where $\mu_0 = 0^\circ C$ is the hypothesised population mean calibration error.

The sample of 64 sensors has an average calibration error of $+0.1883^\circ C$, indicating that the sampled sensors tend to read slightly higher than the reference temperature.


### b.

Interpret the $p$-value as a tail probability conditional on $H_0$ and the assumptions. State the decision and a contextual conclusion without treating the $p$-value as $P(H_0 \mid \mathrm{data})$.


**Answer (b):**

The one-sample t-test produced a two-sided p-value of $0.019031$.

Assuming the null hypothesis ($H_0: \mu = 0$) is true and the test assumptions hold, this means there is approximately a **1.90% probability of obtaining a test statistic as extreme as or more extreme than our observed result ($|t| = 2.4069$)** in either direction.

Since the p-value ($0.019031$) is smaller than the significance level ($\alpha = 0.05$), we **reject $H_0$**.

Therefore, there is statistically significant evidence that the population mean sensor calibration error differs from zero. The positive sample mean ($0.1883^\circ C$) suggests that the sensors tend to read slightly higher than the reference temperature on average.

However, the p-value does not tell us the probability that $H_0$ is true given the data. It only tells us how unusual our test result would be if $H_0$ were true, assuming the test conditions hold.


### c.

Reuse or reconstruct the 95% $t$ interval and explain its duality with this two-sided test. Compare it with the practical limits and distinguish statistical significance from practical importance.


**Answer (c): 95% Confidence Interval**

To estimate the population mean sensor calibration error, we construct a 95% confidence interval using the one-sample t-distribution.

The one-sample t-statistic is:

$$
T = \frac{\bar{X} - \mu_0}{S/\sqrt{n}}
$$

The corresponding 95% t-confidence interval is:

$$
CI = \bar{x} \pm t_{1-\alpha/2,\;n-1}\frac{s}{\sqrt{n}}
$$

where:

- $\bar{x}$ = sample mean
- $s$ = sample standard deviation
- $n$ = sample size
- $\alpha$ = significance level
- $t_{1-\alpha/2,\;n-1}$ = critical t-value

**Using our sample statistics:**

$$
n = 64,\quad \bar{x} = 0.1883,\quad s = 0.6257
$$

$$
\alpha = 0.05,\quad df = 63
$$

Substituting into the confidence interval formula:

$$
CI = 0.1883 \pm t_{0.975,63}\frac{0.6257}{\sqrt{64}}
$$


In [15]:

alpha = 0.05
df = n - 1

standard_error = sample_std / np.sqrt(n)

t_critical = stats.t.ppf(1 - alpha / 2, df)

# Confidence interval
ci_lower = sample_mean - t_critical * standard_error
ci_upper = sample_mean + t_critical * standard_error

print(f"Standard error: {standard_error:.5f}")
print(f"Critical t-value: {t_critical:.4f}")
print(f"95% confidence interval: [{ci_lower:.4f}, {ci_upper:.4f}] °C")


Standard error: 0.07822
Critical t-value: 1.9983
95% confidence interval: [0.0320, 0.3446] °C



**Interpretation:**

The 95% confidence interval for the population mean sensor calibration error is approximately:

$$
CI = [0.032,\ 0.345]^\circ C
$$

This interval provides a range of plausible values for the true population mean calibration error.

**Test–interval duality:**

Since the 95% confidence interval does not contain zero, we reject $H_0: \mu = 0$ at the 5% significance level. This is consistent with our earlier two-sided t-test result ($p = 0.019031 < 0.05$).

**Statistical significance vs. practical importance:**

Although the mean calibration error is statistically significantly different from zero, the entire confidence interval lies within the practical limits of $\pm 0.50^\circ C$.

Therefore, the results suggest that the population mean calibration error is small in practical engineering terms, despite being statistically significant. However, this does not imply that every individual sensor has a small calibration error.


### False-Rejection Probability

Use `assignment03_access_control.csv`. The pre-specified question is whether the false-rejection probability exceeds the 5% target.

### a.

State the one-sided hypotheses. Perform an exact binomial test and report $\hat{p}$, the $p$-value, decision, and conclusion.


**Answer (a):**

Let $p$ be the true false-rejection probability.

$$
H_0: p = 0.05
$$

$$
H_1: p > 0.05
$$

We use a one-sided exact binomial test with $\alpha = 0.05$.

The sample proportion is calculated as:

$$
\hat{p} = \frac{x}{n}
$$

where $x$ is the number of false rejections and $n$ is the total number of attempts.


In [18]:

# False-rejection probability
x = access["false_reject"].sum()
n_access = access["false_reject"].count()
p_hat = x / n_access

# Exact one-sided binomial test
result = stats.binomtest(x, n_access, p=0.05, alternative="greater")

print("False rejections:", x)
print("Total attempts:", n_access)
print(f"Sample proportion: {p_hat:.4f}")
print(f"P-value: {result.pvalue:.6f}")

if result.pvalue < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


False rejections: 20
Total attempts: 240
Sample proportion: 0.0833
P-value: 0.018486
Decision: Reject H0



**Interpretation:**

The observed false-rejection rate is $\hat{p} = 20/240 = 0.0833$ (8.33%).

Since the p-value ($0.018486$) is below $\alpha = 0.05$, we reject $H_0$.

There is sufficient statistical evidence that the true false-rejection probability exceeds the 5% target.


### b.

Use the Wilson interval as an estimate of plausible values, while explaining why a two-sided Wilson interval is not the exact dual of a one-sided exact test.

### c.

Describe Type I and Type II errors in one of the two one-sample studies and state an operational consequence of each.


**Answer (c):Access-control study**

**Type I error ($\alpha$):** We conclude that the false-rejection probability exceeds 5% when it is actually 5%. This could lead to unnecessary system maintenance or adjustments.

**Type II error ($\beta$):** We fail to detect that the false-rejection probability exceeds 5% when it actually does. This could result in authorised users continuing to experience excessive access denials.


<a id="part-2"></a>

## Part 2 — Independent coating processes

### a.

**Interpretation**

### b.

### c.

### d.

**Interpretation**

### e.

### f.

<a id="part-3"></a>

## Part 3 — Paired building-energy measurements

### a.

### b.

### c.

### d.

**Interpretation**

### e.

<a id="part-4"></a>

## Part 4 — Comparing two independent proportions

<a id="part-5"></a>

## Part 5 — AI-use and conclusion

### AI-use statement

| Tool | What it was used for |
|---|---|
| | |

### Conclusion